Mini Project 2: Resume Analyzer & Career Coach
Scenario

A user uploads their resume.

AI Tasks
Summarize the candidate profile.
Extract:
Skills
Experience
Education
Identify missing skills for a target role.
Calculate resume-job match percentage.
Suggest improvements.
Generate a professional LinkedIn headline.
Generate a cover letter.

In [10]:
candiate_profile_summary = """RAJIV SHARMA

Email: [rajiv.sharma@gmail.com](mailto:rajiv.sharma@gmail.com)
Phone: +91 9876543210
Location: Chennai, India

PROFESSIONAL SUMMARY

Software Engineer with 5 years of experience in developing web applications using React, JavaScript, TypeScript, and Node.js. Experienced in building scalable frontend applications, integrating REST APIs, and collaborating with cross-functional teams. Passionate about cloud technologies and AI-powered applications.

TECHNICAL SKILLS

Frontend: 

* React.js
* JavaScript (ES6+)
* TypeScript
* HTML5
* CSS3
* Redux

Backend:

* Node.js
* Express.js

Database:

* MongoDB
* MySQL

Cloud & DevOps:

* AWS (Basic)
* Docker
* GitHub Actions

Tools:

* Git
* Jira
* Postman

WORK EXPERIENCE

Senior Software Engineer
ABC Technologies Pvt Ltd
Jan 2023 - Present

Responsibilities:

* Developed enterprise React applications serving 100,000+ users.
* Improved application performance by 35%.
* Led frontend architecture discussions.
* Mentored 4 junior developers.
* Integrated multiple third-party APIs.

Software Engineer
XYZ Solutions Pvt Ltd
Jun 2021 - Dec 2022

Responsibilities:

* Built reusable React components.
* Developed REST APIs using Node.js.
* Worked with Agile Scrum teams.
* Reduced bug reports by 20%.

Junior Software Engineer
TechSoft Systems
Jul 2020 - May 2021

Responsibilities:

* Maintained internal web applications.
* Fixed production bugs.
* Assisted in frontend development.

EDUCATION

Bachelor of Technology (Computer Science)
Anna University
2016 - 2020

CERTIFICATIONS

* AWS Cloud Practitioner
* React Advanced Certification

PROJECTS

Insurance Portal

* Developed customer onboarding workflows.
* Implemented document upload module.
* Integrated payment gateway.

Employee Management System

* Built role-based access control.
* Created dashboard using React and Redux.
"""


Target Job Description

In [11]:
target_job_description = """Position: Senior Full Stack Engineer

Requirements:

* 5+ years of software development experience
* Strong React.js expertise
* TypeScript proficiency
* Node.js and Express.js
* Experience with AWS services
* CI/CD pipeline knowledge
* Docker and Kubernetes
* Microservices architecture
* GraphQL
* Unit Testing (Jest, React Testing Library)
* Leadership and mentoring experience
* Agile methodology

Preferred Skills:

* Generative AI
* LangChain
* LLM Applications
* Python
* Vector Databases
* RAG Systems
"""


Resume Summary Prompt

In [12]:
candiate_profile_summary_prompt = """
                         As an AI expert you act as Resume Analyzer & Career Coach your job is provide me
                         the candidate resume summary for given resume in 50 words
                         Candidate profile summary :
                         {candiate_profile_summary}
                         """



In [13]:
import os

In [14]:
from dotenv import load_dotenv, find_dotenv
load_dotenv()

True

In [15]:
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    print("No API key was found!")
elif not api_key.startswith("sk-proj"):
    print("An API key was found, but it doesn't start sk-proj-;")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [16]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model_name="gpt-4o-mini",temperature=0.0)

In [18]:
from langchain_core.prompts import PromptTemplate

In [19]:
prompt = PromptTemplate.from_template(template=candiate_profile_summary_prompt)

In [20]:
# create a simple LCEL chain to take the prompt, pass it to the LLM, enforce response format using the parser
chain = (prompt
           |
         llm
        )

In [21]:
response = chain.invoke({"candiate_profile_summary": candiate_profile_summary})

In [22]:
response

AIMessage(content='Dynamic Software Engineer with 5 years of experience in web application development using React, JavaScript, and Node.js. Proven track record in enhancing application performance, mentoring teams, and integrating APIs. Passionate about cloud technologies and AI, with a strong foundation in Agile methodologies and scalable frontend solutions.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 461, 'total_tokens': 519, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9446817294', 'finish_reason': 'stop', 'logprobs': None}, id='run--93931dc5-4e8c-4f09-bf1a-2f896bd5978d-0', usage_metadata={'input_tokens': 461, 'output_tokens': 58, 'total_tokens': 519, 'input_token_details': {'audio': 0, '

In [23]:
print(response.content)

Dynamic Software Engineer with 5 years of experience in web application development using React, JavaScript, and Node.js. Proven track record in enhancing application performance, mentoring teams, and integrating APIs. Passionate about cloud technologies and AI, with a strong foundation in Agile methodologies and scalable frontend solutions.


In [24]:
from IPython.display import display, Markdown
display(Markdown(response.content))

Dynamic Software Engineer with 5 years of experience in web application development using React, JavaScript, and Node.js. Proven track record in enhancing application performance, mentoring teams, and integrating APIs. Passionate about cloud technologies and AI, with a strong foundation in Agile methodologies and scalable frontend solutions.

Skills Extraction Prompt

In [27]:
from typing import List
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
# Define your desired data structure - like a python data class. This will enforce LLM to retrurn with specific datatype.
class Skills(BaseModel):
    frontend: List[str] = Field(description="List of frontend technologies and frameworks")
    backend: List[str] = Field(description="List of backend technollogies and frameworks")
    cloudL: List[str] = Field(description="List of cloud and DevOps technologies")

parser = JsonOutputParser(pyantic_object=Skills)
format_instruction = parser.get_format_instructions()
print(format_instruction)

Return a JSON object.


In [28]:
format_instruction

'Return a JSON object.'

In [29]:
skill_prompt = """
               Extract technical skills from the Candidate profile summary.
               and generate the response based on the following formatting instructions
               Candidate profile summary:
               {candiate_profile_summary}
               Format Instructions:
               {format_instructions}
               """

In [30]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate(template = skill_prompt, input_variables = ['candiate_profile_summary'], partial_variables={"format_instructions": parser.get_format_instructions()} )

In [32]:
chain = (prompt 
          |
         llm
          |
        parser
        )

In [33]:
skill_response = chain.invoke({"candiate_profile_summary":candiate_profile_summary})

In [34]:
skill_response

{'technical_skills': {'frontend': ['React.js',
   'JavaScript (ES6+)',
   'TypeScript',
   'HTML5',
   'CSS3',
   'Redux'],
  'backend': ['Node.js', 'Express.js'],
  'database': ['MongoDB', 'MySQL'],
  'cloud_and_devops': ['AWS (Basic)', 'Docker', 'GitHub Actions'],
  'tools': ['Git', 'Jira', 'Postman']}}

In [37]:
import pandas as pd

df = pd.DataFrame(skill_response)
df

,technical_skills
frontend,"[React.js, JavaScript (ES6+), TypeScript, HTML..."
backend,"[Node.js, Express.js]"
database,"[MongoDB, MySQL]"
cloud_and_devops,"[AWS (Basic), Docker, GitHub Actions]"
tools,"[Git, Jira, Postman]"


Experience Extraction

In [58]:
class Experience(BaseModel):
    total_experience: str = Field(description="Total experience of candidate in years")
    current_role: str = Field(description="Current role of the candidate")
    companies: List[str] = Field(description="List of companies candiate has worked till now")

exp_parser = JsonOutputParser(pydantic_object=Experience)
print(exp_parser)


pydantic_object=<class '__main__.Experience'>


In [59]:
experience_prompt = """
               Extract candidate experience, current role and previous experinces from the Candidate profile summary.
               and generate the response based on the following formatting instructions
               Candidate profile summary:
               {candiate_profile_summary}
               Format Instructions:
               {exp_format_instructions}
               """

In [68]:
from langchain_core.prompts import PromptTemplate
exp_prompt = PromptTemplate(template = experience_prompt, input_variables = ['candiate_profile_summary'], partial_variables={"exp_format_instructions": exp_parser.get_format_instructions()} )
exp_prompt

PromptTemplate(input_variables=['candiate_profile_summary'], input_types={}, partial_variables={'exp_format_instructions': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"total_experience": {"description": "Total experience of candidate in years", "title": "Total Experience", "type": "string"}, "current_role": {"description": "Current role of the candidate", "title": "Current Role", "type": "string"}, "companies": {"description": "List of companies candiate has worked till now", "items": {"type": "string"}, "title": "Companies", "type": "array"}}, "required": ["tota

In [69]:
chain = (exp_prompt 
          |
         llm
          |
        exp_parser
        )

In [70]:
experience_response = chain.invoke({"candiate_profile_summary":candiate_profile_summary})

In [71]:
experience_response

{'total_experience': '5',
 'current_role': 'Senior Software Engineer',
 'companies': ['ABC Technologies Pvt Ltd',
  'XYZ Solutions Pvt Ltd',
  'TechSoft Systems']}